<a href="https://colab.research.google.com/github/aswiniaswini8930-afk/NutriGuide-AI/blob/main/NutriGuide_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq


In [ ]:
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

print("API key loaded successfully!" if GROQ_API_KEY else "API key not found")

API key loaded successfully!


In [ ]:
from groq import Groq
from google.colab import userdata

# Connect to Groq
client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

# Send our first request
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI nutrition assistant."
        },
        {
            "role": "user",
            "content": "Explain why protein is important for the human body."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

**Protein: Why It’s Essential for Your Body**

| Function | How Protein Helps | Why It Matters |
|----------|-------------------|----------------|
| **Building & repairing tissues** | Amino acids are the “bricks” that form muscles, skin, hair, nails, organs, and even bone matrix. | Keeps you strong, heals wounds faster, and maintains the structural integrity of every cell. |
| **Enzymes & metabolic reactions** | Most enzymes are proteins; they speed up chemical reactions that turn food into energy, synthesize DNA, and break down toxins. | Without enzymes, vital biochemical processes would occur far too slowly to sustain life. |
| **Hormones & signaling molecules** | Many hormones (e.g., insulin, glucagon, growth hormone) are proteins or peptides. | Regulate blood sugar, growth, metabolism, and many other body functions. |
| **Immune system support** | Antibodies (immunoglobulins) are proteins that recognize and neutralize pathogens. | Protects you from infections and helps the body rec

In [ ]:
models = client.models.list()

for model in models.data:
    print(model.id)

whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-120b
whisper-large-v3
qwen/qwen3.8-27b
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
canopylabs/orpheus-arabic-saudi
allam-2-7b
meta-llama/llama-prompt-guard-2-22m
openai/gpt-oss-safeguard-20b


In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI nutrition assistant."
        },
        {
            "role": "user",
            "content": "Explain why protein is important for the human body."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

**Protein is one of the three macronutrients (along with carbohydrates and fats) that our bodies need in relatively large amounts.**  
It plays a central, “Swiss‑army‑knife” role in virtually every physiological process. Below is a concise but comprehensive overview of why protein is essential for human health.

---

## 1. Building Blocks of Cells and Tissues
| Function | How Protein Contributes |
|----------|--------------------------|
| **Structural support** | Collagen, elastin, keratin, and other structural proteins give strength and elasticity to skin, bones, tendons, hair, and nails. |
| **Muscle mass** | Actin and myosin filaments in muscle fibers are proteins; they contract to produce movement and generate strength. |
| **Organs & membranes** | Cell membranes contain integral proteins (e.g., receptors, channels) that maintain shape, transport substances, and mediate cell‑to‑cell communication. |

> **Bottom line:** Without adequate protein, the body cannot build or maintain the

In [ ]:
import pandas as pd

food_data = {
    "food": ["boiled egg", "milk", "banana", "rice", "chicken breast"],
    "serving_g": [50, 100, 100, 100, 100],
    "calories": [78, 61, 89, 130, 165],
    "protein": [6.3, 3.2, 1.1, 2.7, 31],
    "carbs": [0.6, 4.8, 22.8, 28, 0],
    "fat": [5.3, 3.3, 0.3, 0.3, 3.6]
}

food_df = pd.DataFrame(food_data)

food_df

,food,serving_g,calories,protein,carbs,fat
0,boiled egg,50,78,6.3,0.6,5.3
1,milk,100,61,3.2,4.8,3.3
2,banana,100,89,1.1,22.8,0.3
3,rice,100,130,2.7,28.0,0.3
4,chicken breast,100,165,31.0,0.0,3.6


In [ ]:
def get_food_nutrition(food_name):

    result = food_df[
        food_df["food"].str.lower() == food_name.lower()
    ]

    if result.empty:
        return {"error": f"{food_name} not found in database"}

    food = result.iloc[0]

    return {
        "food": food["food"],
        "serving_g": int(food["serving_g"]),
        "calories": float(food["calories"]),
        "protein": float(food["protein"]),
        "carbs": float(food["carbs"]),
        "fat": float(food["fat"])
    }

In [ ]:
get_food_nutrition("boiled egg")

{'food': 'boiled egg',
 'serving_g': 50,
 'calories': 78.0,
 'protein': 6.3,
 'carbs': 0.6,
 'fat': 5.3}

In [ ]:
get_food_nutrition("banana")

{'food': 'banana',
 'serving_g': 100,
 'calories': 89.0,
 'protein': 1.1,
 'carbs': 22.8,
 'fat': 0.3}

In [ ]:
import json

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_food_nutrition",
            "description": "Get nutrition information for a food from our food database.",
            "parameters": {
                "type": "object",
                "properties": {
                    "food_name": {
                        "type": "string",
                        "description": "Name of the food, such as banana or boiled egg"
                    }
                },
                "required": ["food_name"]
            }
        }
    }
]

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a nutrition assistant. Use the get_food_nutrition tool whenever the user asks for food nutrition values."
    },
    {
        "role": "user",
        "content": "How much protein is present in 100g of banana?"
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning="We need to use get_food_nutrition tool to get nutrition info for banana. Then extract protein per 100g. Probably the tool returns nutrition per 100g. We'll call it.", tool_calls=[ChatCompletionMessageToolCall(id='fc_965f29fb-fd7e-46a3-b1f4-5cd442c8ad48', function=Function(arguments='{"food_name":"banana"}', name='get_food_nutrition'), type='function')])


In [ ]:
import json

# Add Groq's tool-call message to conversation
messages.append(response.choices[0].message)

# Execute the requested Python function
for tool_call in response.choices[0].message.tool_calls:

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_food_nutrition":

        result = get_food_nutrition(**arguments)

        print("Tool executed!")
        print("Tool result:", result)

        # Send the result back to Groq
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result)
        })

# Ask Groq to generate the final answer
final_response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=tools
)

print("\nAgent's final answer:")
print(final_response.choices[0].message.content)

Tool executed!
Tool result: {'food': 'banana', 'serving_g': 100, 'calories': 89.0, 'protein': 1.1, 'carbs': 22.8, 'fat': 0.3}

Agent's final answer:
A 100‑gram serving of banana contains **approximately 1.1 grams of protein**.


In [ ]:
def nutrition_agent(user_question):

    messages = [
        {
            "role": "system",
            "content": """
            You are NutriGuide AI, a nutrition assistant.
            Use the nutrition database tool whenever food
            nutrition values are requested.
            Never invent nutritional values.
            """
        },
        {
            "role": "user",
            "content": user_question
        }
    ]

    # Ask Groq to decide whether a tool is needed
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # Execute requested tools
    if message.tool_calls:

        messages.append(message)

        for tool_call in message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(
                tool_call.function.arguments
            )

            if function_name == "get_food_nutrition":

                result = get_food_nutrition(**arguments)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result)
                })

        # Get final answer
        final_response = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=messages,
            tools=tools
        )

        return final_response.choices[0].message.content

    return message.content

In [ ]:
print(nutrition_agent("What are the nutrients in milk?"))

Here’s a typical nutrient profile for **100 g of plain cow’s milk** (often used as a reference amount):

| Nutrient (per 100 g) | Amount |
|----------------------|--------|
| **Energy (Calories)** | 61 kcal |
| **Protein** | 3.2 g |
| **Carbohydrates** | 4.8 g (mostly lactose) |
| **Fat** | 3.3 g (includes saturated, monounsaturated, and a small amount of polyunsaturated fats) |
| **Calcium** | ~120 mg |
| **Vitamin D** | ~0.1 µg (often fortified in commercial milks) |
| **Vitamin B12** | ~0.5 µg |
| **Riboflavin (B2)** | ~0.18 mg |
| **Phosphorus** | ~95 mg |
| **Potassium** | ~150 mg |
| **Sodium** | ~44 mg |
| **Magnesium** | ~10 mg |
| **Vitamin A** | ~46 µg (retinol activity equivalents) |
| **Sugar (lactose)** | ~4.8 g |
| **Water** | ~87 g |

*Note:* The exact composition can vary slightly depending on the type of milk (whole, 2 % reduced‑fat, skim, or fortified varieties). If you’re interested in a specific kind of milk (e.g., skim, fortified, or plant‑based alternatives), let 